# Technical Analysis Evolution Lab — Colab Launcher

Research-only runner. The final holdout is kept blind until development selection is complete.

In [ ]:
import os, sys, subprocess, yaml, json
REPO='https://github.com/betaanoiar1-gif/Technical-Analysis-Evolution-Lab.git'
WORK='/content/Technical-Analysis-Evolution-Lab'
if not os.path.exists(WORK): subprocess.run(['git','clone','--depth','1',REPO,WORK], check=True)
%cd $WORK
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)
sys.path.insert(0, os.path.join(WORK, 'src'))
import taevo
print('TAEVO package:', taevo.__version__)


In [ ]:
SYMBOL='BTC/USDT'
TIMEFRAME='1h'
EXCHANGE='binance'
LIMIT=3000
CONFIG_PATH='configs/default.yaml'
DATA_MODE='exchange'  # exchange or upload


In [ ]:
from taevo.data import fetch_exchange, load_csv
if DATA_MODE=='upload':
    from google.colab import files
    uploaded=files.upload()
    bundle=load_csv(next(iter(uploaded)), SYMBOL, TIMEFRAME)
else:
    bundle=fetch_exchange(SYMBOL, TIMEFRAME, LIMIT, EXCHANGE)
print(bundle.symbol, bundle.timeframe, bundle.source, len(bundle.frame), bundle.fingerprint[:16])
display(bundle.frame.tail())


In [ ]:
from taevo.experiment import run_experiment
from taevo.reporting import render_html
cfg=yaml.safe_load(open(CONFIG_PATH, encoding='utf-8'))
report=run_experiment(bundle, cfg, 'experiments/runs')
accepted=report['best_accepted_development']
if not accepted: raise RuntimeError('No candidate passed development validation. This is a valid research outcome.')
best=accepted[0]
print('Selected development candidate:', best['strategy'])
print('Validation score:', round(best['validation']['score'],4))
print('Holdout return:', round(100*best['holdout']['total_return'],2),'%')
print('Holdout buy-and-hold:', round(100*best['holdout_buy_hold']['total_return'],2),'%')
render_html(report,'reports/latest.html')


In [ ]:
import matplotlib.pyplot as plt
from taevo.backtest import run_backtest
from taevo.data import split_three_way
from taevo.strategies import strategy_from_record
_,_,holdout=split_three_way(bundle.frame,cfg['validation']['train_ratio'],cfg['validation']['validation_ratio'])
strategy=strategy_from_record(best)
result=run_backtest(holdout,strategy.signal(holdout),cfg['capital']['initial_usd'],cfg['costs']['fee_bps'],cfg['costs']['slippage_bps'])
result.equity.plot(figsize=(12,4),title='Selected candidate — blind HOLDOUT equity')
plt.ylabel('Equity'); plt.show()


In [ ]:
from google.colab import files
files.download('reports/latest.html')
